In [25]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision.transforms import v2
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from sklearn.preprocessing import label_binarize
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import LeaveOneGroupOut
from torchinfo import summary
from scipy import signal
import pandas as pd
import numpy as np
import warnings
import math
import time
import os
warnings.filterwarnings('ignore')

In [26]:
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 150)

In [27]:
while not os.path.isdir(os.path.join(os.getcwd(), 'data')):
    os.chdir("../") # set cwd to root dir

**Enable CUDA if available**

In [28]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

device

NVIDIA GeForce RTX 3050 Ti Laptop GPU


device(type='cuda')

**Custom Transforms**

In [ ]:
class RandomScaling(object):
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, input):
        output = input * torch.normal(mean=self.mean, std=self.std, size=())
        return output

In [29]:
class ButterworthFilter(object):
    def __init__(self, cutoff, order, fs):
        self.cutoff = cutoff
        self.order = order
        self.fs = fs

    def __call__(self, input): # input must be numpy array
        nyquist = 0.5 * self.fs
        normal_cutoff = self.cutoff / nyquist
        b, a = signal.butter(self.order, normal_cutoff, btype='low', analog=False)
        smoothed_imu_signal = signal.filtfilt(b, a, input)

        return torch.from_numpy(smoothed_imu_signal.copy()) # convert back to tensor

In [30]:
class PadTrimToLength(object):
    def __init__(self, padlen):
        self.padlen = padlen

    def __call__(self, input):
        pad = nn.ZeroPad1d((0, max(0, self.padlen - input.shape[1])))(input)
        output = torch.narrow(pad, 1, 0, self.padlen)
        assert output.shape[1] == self.padlen, "Not equal to length of pad"
        return output

In [31]:
class Normalize(object):
    def __init__(self, mean, std, epsilon=1e-7):
        self.mean = mean
        self.std = std
        self.epsilon = epsilon

    def __call__(self, input):
        output = (input - self.mean) / (self.std + self.epsilon)
        return output

**Define dataset pipeline**

In [32]:
class DUO_GAIT(Dataset):
    def __init__(self, freq, allowed_sensors, num_participants, ignore_participant_ids, compute_jerk=True, include_quat=True, include_dt=True, remove_outliers=True, transform=None, target_transform=None):
        self.num_participants = num_participants
        self.allowed_sensors = allowed_sensors
        self.foot_strides_df = None
        self.imu_signals = []
        self.freq = freq

        sensor_dict = {}
        for participant_id in range(1, num_participants+1):
            if participant_id in ignore_participant_ids:
                continue

            for sensor_location in self.allowed_sensors:
                for protocol in ["control", "fatigue"]:
                    for task in ["st", "dt"]:
                        if task == "dt" and not include_dt: # skip dual task if we don't want it
                            continue

                        interim_file_name = f"data/DUO-GAIT/interim/OG_{task}_{protocol}/sub_{participant_id:02}/{sensor_location}.csv"
                        interim_sensor_df = pd.read_csv(interim_file_name)
                        interim_sensor_df.rename({ "timestamp": "Time (secs)", "Unnamed: 0": "Sample" }, axis=1, inplace=True)

                        if include_quat:
                            raw_file_name = f"data/DUO-GAIT/raw/OG_{task}_raw/sub_{participant_id:02}/{sensor_location}.csv"
                            raw_sensor_df = pd.read_csv(raw_file_name,skiprows=5).drop(index=0).astype(float)
                            raw_sensor_df = raw_sensor_df.filter(["Quat W", "Quat X", "Quat Y", "Quat Z"], axis=1)
                            raw_sensor_df.rename({ "Quat W": "QuatW", "Quat X": "QuatX", "Quat Y": "QuatY", "Quat Z": "QuatZ" },axis=1, inplace=True, errors='raise')

                            quat_features_df = raw_sensor_df.loc[interim_sensor_df['Sample'].min(): interim_sensor_df['Sample'].max()+1].reset_index(drop=True) # get quat
                            interim_sensor_df = pd.merge(interim_sensor_df, quat_features_df, left_index=True, right_index=True, how='inner')

                        sensor_dict[interim_file_name] = interim_sensor_df
            # save interim files for future

            for protocol in ["control", "fatigue"]:
                for foot in ["left", "right"]:
                    for task in ["st", "dt"]:
                        if task == "dt" and not include_dt: # skip dual task if we don't want it
                            continue

                        foot_df = pd.read_csv(f"data/DUO-GAIT/processed/OG_{task}_{protocol}/sub_{participant_id:02}/{foot}_foot_core_params.csv")
                        foot_df.rename({ "timestamps": "start_times" }, axis=1, inplace=True)
                        foot_df['is_fatigue'] = int(protocol == "fatigue")
                        foot_df['is_dual_task'] = int(task == "dt")
                        foot_df['Participant'] = participant_id

                        is_control = (protocol == "control")
                        is_dual_task = (task == "dt")
                        
                        self.create_start_end_samples_strides(sensor_dict, foot_df, participant_id, is_control=is_control, is_dual_task=is_dual_task)
                        self.foot_strides_df = pd.concat([self.foot_strides_df, foot_df], axis=0)

        if remove_outliers:
            self.foot_strides_df = self.foot_strides_df[self.foot_strides_df['is_outlier']==False].reset_index(drop=True)

        for _, row in self.foot_strides_df.iterrows():
            imu_signals_df_list = []
            for sensor_location in self.allowed_sensors:
                protocol = "fatigue" if row['is_fatigue'] else "control"
                task = "dt" if row['is_dual_task'] else "st"
                file_name = f"data/DUO-GAIT/interim/OG_{task}_{protocol}/sub_{row['Participant']:02}/{sensor_location}.csv"

                sensor_df = sensor_dict[file_name]
                sensor_df = sensor_df[(sensor_df['Sample'] >= row['start_samples']) & (sensor_df['Sample'] <= row['end_samples'])]

                acc_columns = ["AccX", "AccY", "AccZ"]
                gyr_columns = ["GyrX", "GyrY", "GyrZ"]
                quat_columns = ["QuatW", "QuatX", "QuatY", "QuatZ"] if include_quat else []

                sensor_signals_df = sensor_df[acc_columns + gyr_columns + quat_columns].reset_index(drop=True)
                sensor_signals_df = sensor_signals_df.add_prefix(f"{sensor_location}_")

                if compute_jerk:
                    acc_gyro_df = sensor_df[acc_columns + gyr_columns]
                    acc_gyro_np = acc_gyro_df.to_numpy().transpose().astype(np.float32)
                    jerk_np = np.gradient(acc_gyro_np, 1.0 / self.freq, axis=1).transpose()
                    jerk_df = pd.DataFrame(data=jerk_np, columns=["Jerk" + col for col in (acc_columns + gyr_columns)])
                    sensor_signals_df = pd.concat([sensor_signals_df, jerk_df], axis=1)

                imu_signals_df_list.append(sensor_signals_df)

            imu_signals_df = pd.concat(imu_signals_df_list, axis=1)
            imu_signals_np = imu_signals_df.to_numpy().transpose().astype(np.float32)
            self.imu_signals.append(imu_signals_np)

        self.num_channels = self.imu_signals[0].shape[0]
        self.transform = transform
        self.target_transform = target_transform

    def create_start_end_samples_strides(self, sensor_dict, df, participant_id, is_control, is_dual_task):
        df.sort_values(by='stride_index', inplace=True)
        df['start_samples'] = df['ic_samples'].shift(1)
        target_time = df.loc[0, 'start_times']
        
        protocol = "control" if is_control else "fatigue"
        task = "dt" if is_dual_task else "st"

        file_name = f"data/DUO-GAIT/interim/OG_{task}_{protocol}/sub_{participant_id:02}/{self.allowed_sensors[0]}.csv"        
        fatigue_df = sensor_dict[file_name].copy() # copy to avoid changing sensor dict's dataframes
        fatigue_df['Delta (secs)'] = fatigue_df['Time (secs)'] - fatigue_df['Time (secs)'].min() # delta time

        ts_eq_check = fatigue_df['Delta (secs)'].apply(lambda x: math.isclose(x, target_time, rel_tol=1e-5))
        start_sample = fatigue_df[ts_eq_check]['Sample'].item() - fatigue_df['Sample'].min()

        df.loc[0, 'start_samples'] = start_sample
        df['start_samples'] = df['start_samples'] + fatigue_df['Sample'].min()
        df['end_samples'] = df['ic_samples'] + fatigue_df['Sample'].min() - 1 # make it inclusive for ending samples too

        df['start_samples'] = df['start_samples'].astype(np.int64)
        df['end_samples'] = df['end_samples'].astype(np.int64)

    def __len__(self):
        return len(self.foot_strides_df)

    def __getitem__(self, idx):
        row = self.foot_strides_df.iloc[idx]

        imu_signals_np = self.imu_signals[idx]
        label = row["is_fatigue"]

        if self.transform:
            imu_signals_np = self.transform(imu_signals_np)

        if self.target_transform:
            label = self.target_transform(label)

        return imu_signals_np, label

**CNN Architecture**

In [33]:
class FatigueCNN(nn.Module):
    def __init__(self, p_drop):
        super().__init__()

        self.features = nn.Sequential(
                        nn.LazyConv1d(out_channels=32, kernel_size=5, stride=1, padding=0),
                        nn.ReLU())

        self.gap = nn.AdaptiveMaxPool1d(output_size=1)
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(p=p_drop)
        self.linear = nn.Linear(in_features=32, out_features=2)

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.linear(x)
        
        return x

**Experiment Parameters**

In [34]:
num_participants = 18
ignore_participant_ids = [1, 4, 7, 16]
allowed_sensors = ["LL", "RL"]
num_stride_samples = 160

**Hyper Parameters**

In [35]:
batch_size = 32
epochs = 50 # TODO - change this to infinite and implement early stopping instead
learning_rate = 0.0005 # TODO - consider lowering LR to gradually improve accuracy over the epochs
min_learning_rate = 0.0001
patience_epochs = 15
learning_rate_factor = 0.8
early_stopping_rounds = 20
train_percent = 0.90
p_drop = 0.5

**Leave one Participant out Training Loop**

In [ ]:
print("Loading Dataset ...")
dataset = DUO_GAIT(128.0, allowed_sensors, num_participants, ignore_participant_ids, include_dt=False)
print("Dataset Loaded ...")

dataset_size = len(dataset)
groups = dataset.foot_strides_df['Participant'].to_numpy()

gss = GroupShuffleSplit(n_splits=1, train_size=train_percent, random_state=42)
train_val_idx, test_idx = next(gss.split(X=dataset, groups=groups))

train_val_set = Subset(dataset, train_val_idx)
test_set = Subset(dataset, test_idx)

groups_train_val = groups[train_val_idx]

train_lopo_accuracies = []
valid_lopo_accuracies = []

logo = LeaveOneGroupOut()
for i, (train_index, valid_index) in enumerate(logo.split(X=train_val_set, groups=groups_train_val)):
    train_set = Subset(train_val_set, train_index)
    valid_set = Subset(train_val_set, valid_index)

    dataset.transform = v2.Compose([
        ButterworthFilter(cutoff=10.0, order=3, fs=128.0),
        RandomScaling(mean=1.0,std=0.2),
        PadTrimToLength(padlen=num_stride_samples),
    ]) # default transform before normalization

    stat_loader = DataLoader(train_set, batch_size=batch_size)
    lopo = groups_train_val[valid_index][0]

    full_train_data = []
    for batch_idx, (train_features, train_labels) in enumerate(stat_loader):
        full_train_data.append(train_features)

    full_train_data = torch.concat(full_train_data, dim=0)
    std, mean = torch.std_mean(full_train_data, dim=(0, 2), keepdim=True)
    # compute mean and std over all channels separately

    std = torch.squeeze(std, dim=0)
    mean = torch.squeeze(mean, dim=0)
    # remove the batch dimension

    dataset.transform = v2.Compose([
        ButterworthFilter(cutoff=10.0, order=3, fs=128.0),
        RandomScaling(mean=1.0,std=0.2),
        PadTrimToLength(padlen=num_stride_samples),
        Normalize(mean=mean, std=std),
        v2.Lambda(lambda x: x.to(torch.float32))
    ]) # update transform for normalization

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, pin_memory=True)
    valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=False, pin_memory=True)
    # create dataloaders for training and validation

    timestamp = time.strftime("%b-%d-%Y %I-%M-%S %p")
    writer = SummaryWriter(log_dir=f'runs/{timestamp}/lopo_{lopo:02} p_drop_{p_drop} lr_{learning_rate} {' '.join(allowed_sensors)} 1_conv jerk quat', flush_secs=30)

    model = FatigueCNN(p_drop).to(device=device) # move model to device

    loss = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=learning_rate_factor, patience=patience_epochs, min_lr=min_learning_rate)

    epochs_without_gain = 0
    best_valid_loss = float("inf")

    max_train_acc = 0
    max_valid_acc = 0

    print(f"Beginning training for lopo_{lopo:02}")
    for epoch in range(0, epochs):
        train_epoch_loss, valid_epoch_loss = 0.0, 0.0

        train_epoch_preds, train_epoch_labels = [], []
        valid_epoch_preds, valid_epoch_labels = [], []

        train_start_time = time.time()
        model.train()
        for batch_idx, (train_features, train_labels) in enumerate(train_loader):
            train_batch_size = len(train_labels)

            train_features = train_features.to(device)
            train_labels = train_labels.to(device) # move data to device

            optimizer.zero_grad()

            logits = model(train_features)
            predictions = torch.argmax(logits, dim=1)
            
            train_epoch_preds.extend(predictions.cpu().numpy())
            train_epoch_labels.extend(train_labels.cpu().numpy())

            train_batch_loss = loss(logits, train_labels)
            train_batch_loss.backward()

            optimizer.step()

            train_epoch_loss += train_batch_loss.item() * train_batch_size
        train_end_time = time.time()

        model.eval()
        valid_start_time = time.time()
        with torch.no_grad():
            for batch_idx, (valid_features, valid_labels) in enumerate(valid_loader):
                valid_batch_size = len(valid_labels)
                
                valid_features = valid_features.to(device)
                valid_labels = valid_labels.to(device) # move to device
                
                logits = model(valid_features)
                predictions = torch.argmax(logits, dim=1)

                valid_epoch_preds.extend(predictions.cpu().numpy())
                valid_epoch_labels.extend(valid_labels.cpu().numpy())

                valid_batch_loss = loss(logits, valid_labels)
                valid_epoch_loss += valid_batch_loss.item() * valid_batch_size
        valid_end_time = time.time()

        train_epoch_loss /= len(train_set)
        valid_epoch_loss /= len(valid_set)

        scheduler.step(valid_epoch_loss) # step the scheduler and reduce lr if necessary

        train_epoch_acc = accuracy_score(train_epoch_labels, train_epoch_preds)
        valid_epoch_acc = accuracy_score(valid_epoch_labels, valid_epoch_preds)

        max_train_acc = max(max_train_acc, train_epoch_acc)
        max_valid_acc = max(max_valid_acc, valid_epoch_acc)

        writer.add_scalar("Loss/train-epoch", train_epoch_loss, epoch)
        writer.add_scalar('Accuracy/train-epoch', train_epoch_acc, epoch)
        writer.add_scalar('Elapsed Time/train-epoch-secs', train_end_time - train_start_time, epoch)

        writer.add_scalar("Loss/valid-epoch", valid_epoch_loss, epoch)
        writer.add_scalar('Accuracy/valid-epoch', valid_epoch_acc, epoch)
        writer.add_scalar('Elapsed Time/valid-epoch-secs', valid_end_time - valid_start_time, epoch)

        output_dir = f"checkpoints/lopo_{lopo:02}"
        os.makedirs(output_dir, exist_ok=True) # create checkpoints dirs

        torch.save(model.state_dict(), f"{output_dir}/epoch_{epoch+1}.pth") # checkpoint for safety

        if valid_epoch_loss < best_valid_loss:
            best_valid_loss = valid_epoch_loss
            epochs_without_gain = 0
        else:
            epochs_without_gain += 1

        if epochs_without_gain >= early_stopping_rounds:
            break

    train_lopo_accuracies.append(max_train_acc)
    valid_lopo_accuracies.append(max_valid_acc)

train_lopo_accuracies = np.array(train_lopo_accuracies)
valid_lopo_accuracies = np.array(valid_lopo_accuracies)

print("Train Acc:", np.mean(train_lopo_accuracies), "+=", np.std(train_lopo_accuracies))
print("Valid Acc:", np.mean(valid_lopo_accuracies), "+=", np.std(valid_lopo_accuracies))

Loading Dataset ...
Dataset Loaded ...
Beginning training for lopo_02
Beginning training for lopo_03
Beginning training for lopo_05
Beginning training for lopo_06


KeyboardInterrupt: 